# 04 — BQ2: Which Early Behavioral Signals Predict Drop-Out?

> **Notebook 04 of 7** | Learning Retention Analytics  
> Second business question analysis: identifying early engagement signals that differentiate completers from non-completers, with formal statistical testing.

## Purpose and Scope

This notebook answers **BQ2: Which early behavioral signals predict drop-out?** — the second of five business questions driving this project.

Notebook 02 (EDA) showed that engagement metrics *differ* between completers and non-completers. Notebook 03 (BQ1) established *when* students leave. This notebook moves from observation to **formal statistical testing**: which early signals (first 28 days) are statistically significant predictors of eventual dropout, and how strong are the effects?

**What this notebook covers:**
- Loading and inspecting the BQ2 analysis dataset (`q_bq2_early_signals.sql`)
- Descriptive comparison of 8 early signals by completion outcome
- Independent t-tests with Cohen's d effect sizes on each signal
- Multiple comparison correction (Bonferroni and Benjamini-Hochberg)
- Forest plot of ranked effect sizes — the key deliverable figure
- Signal deep-dives: dose-response for the strongest predictors
- Assessment-based signals: first score and submission timing
- Ghost student signal quantification with bootstrap confidence intervals

**What this notebook does NOT do:**
- No machine learning models. All analysis uses descriptive and inferential statistics.
- No causal claims. We identify *associations*, not causes.

**What comes next:**
- **Notebook 05** (`05_bq3_demographics_vs_behavior.ipynb`): demographics vs. behavior — what predicts outcome more? (BQ3)

> **Methodological transferability:** The signal-ranking approach used here — systematic t-tests with effect sizes, multiple comparison correction, and forest plots — is the standard toolkit for identifying predictive features in product analytics. Replace "engagement signals" with "feature usage metrics" and "completion" with "retention" to apply this framework to SaaS churn analysis, subscription renewal prediction, or fitness app engagement scoring.

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [The Analysis Dataset](#2.-The-Analysis-Dataset)
3. [Descriptive Comparison](#3.-Descriptive-Comparison)
4. [Statistical Testing — T-Tests](#4.-Statistical-Testing-—-T-Tests)
5. [Multiple Comparison Correction](#5.-Multiple-Comparison-Correction)
6. [Effect Size Ranking — Forest Plot](#6.-Effect-Size-Ranking-—-Forest-Plot)
7. [Signal Deep-Dives](#7.-Signal-Deep-Dives)
8. [Assessment-Based Signals](#8.-Assessment-Based-Signals)
9. [Ghost Student Signal](#9.-Ghost-Student-Signal)
10. [Key Takeaways and Next Steps](#10.-Key-Takeaways-and-Next-Steps)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer (ADR-003).
- BQ2's primary SQL query lives in `sql/queries/q_bq2_early_signals.sql` and is loaded at runtime from disk.
- Statistical tests use `src.stats.tests` — project wrappers around scipy that standardize output format (test statistic, p-value, effect size, confidence interval).
- Figures are saved to `reports/figures/` at 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query
from src.stats.tests import (
    apply_multiple_comparison_correction,
    bootstrap_ci,
    independent_t_test,
)

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE_OUTCOME = {
    'Pass': '#4C72B0',
    'Distinction': '#55A868',
    'Fail': '#C44E52',
    'Withdrawn': '#8172B3',
}
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Shared axis labels
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'
LABEL_EFFECT_SIZE = "Cohen's d"

# Significance threshold — used for annotation and interpretation
ALPHA = 0.05

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ2 query from SQL file ---
_bq2_sql_path = QUERIES_DIR / 'q_bq2_early_signals.sql'
BQ2_SQL = _bq2_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ2 query from: {_bq2_sql_path.name} ({len(BQ2_SQL):,} chars)')

# --- Prerequisite check ---
try:
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_student = _check_student['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    if _n_student == 0 or _n_early == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. The Analysis Dataset

The BQ2 query (`q_bq2_early_signals.sql`) joins `v_student_enriched` with `v_engagement_early` and first-assessment data to create a single analysis-ready DataFrame. Each row represents one student-enrollment with:

| Column | Description | Source |
|--------|-------------|--------|
| `active_days_first_28` | Distinct days with VLE activity (0–28) | `v_engagement_early` |
| `total_clicks_first_28` | Total clicks in first 28 days | `v_engagement_early` |
| `avg_clicks_per_active_day` | Click intensity per active day | `v_engagement_early` |
| `last_active_day_in_window` | Last day of activity within window | `v_engagement_early` |
| `engagement_decile_in_course` | Within-course engagement rank (1–10) | `v_engagement_early` |
| `first_score` | Score on first assessment (NULL if none) | `studentAssessment` |
| `first_submit_day` | Submission day of first assessment (NULL if none) | `studentAssessment` |
| `date_registration` | Registration day (negative = early registration) | `v_student_enriched` |
| `completed` | Binary outcome: 1 = completed, 0 = not completed | `v_student_enriched` |

Students with zero VLE activity get `COALESCE` values of 0 for engagement metrics — they are included as the extreme low end of the engagement spectrum.

In [ ]:
# --- Load BQ2 analysis dataset ---
df_signals = execute_query(BQ2_SQL)

print(f'BQ2 dataset: {len(df_signals):,} rows x {df_signals.shape[1]} columns')
print('\nOutcome distribution:')
print(df_signals['completed'].value_counts().rename(LABEL_BINARY).to_string())

# --- Signal columns to test ---
# These are the 8 early behavioral signals we will compare between
# completed (1) and not-completed (0) groups.
SIGNAL_COLUMNS = [
    'active_days_first_28',
    'total_clicks_first_28',
    'avg_clicks_per_active_day',
    'last_active_day_in_window',
    'engagement_decile_in_course',
    'first_score',
    'first_submit_day',
    'date_registration',
]

# Readable labels for plots
SIGNAL_LABELS = {
    'active_days_first_28': 'Active days (first 28)',
    'total_clicks_first_28': 'Total clicks (first 28)',
    'avg_clicks_per_active_day': 'Avg clicks per active day',
    'last_active_day_in_window': 'Last active day in window',
    'engagement_decile_in_course': 'Engagement decile',
    'first_score': 'First assessment score',
    'first_submit_day': 'First submission day',
    'date_registration': 'Registration day',
}

# --- Missingness check ---
# first_score and first_submit_day are NULL for students who never submitted
# an assessment in the first 28 days. The t-test wrapper handles NaN-dropping,
# but we need to report the effective sample size for each signal.
print('\n=== Non-null counts per signal ===')
for col in SIGNAL_COLUMNS:
    n_valid = df_signals[col].notna().sum()
    n_missing = df_signals[col].isna().sum()
    print(f'  {SIGNAL_LABELS[col]:35s} {n_valid:>8,} valid   {n_missing:>6,} missing')

## 3. Descriptive Comparison

Before statistical testing, we visually compare the distribution of each signal between the two outcome groups. This builds intuition about *where* the differences lie and whether the distributions overlap substantially.

Violin plots show the full distribution shape — more informative than simple bar charts of means, especially for detecting bimodality or heavy tails common in clickstream data.

In [ ]:
# --- Summary statistics by outcome group ---
df_signals['outcome'] = df_signals['completed'].map(LABEL_BINARY)

# Display mean and median for each signal by group
summary_rows = []
for col in SIGNAL_COLUMNS:
    for outcome_val in [1, 0]:
        subset = df_signals[df_signals['completed'] == outcome_val][col].dropna()
        summary_rows.append({
            'Signal': SIGNAL_LABELS[col],
            'Outcome': LABEL_BINARY[outcome_val],
            'N': len(subset),
            'Mean': round(subset.mean(), 2),
            'Median': round(subset.median(), 2),
            'Std': round(subset.std(), 2),
        })

df_summary = pd.DataFrame(summary_rows)
print('=== Descriptive Statistics by Outcome ===\n')
# Pivot for side-by-side comparison
for col in SIGNAL_COLUMNS:
    label = SIGNAL_LABELS[col]
    row_data = df_summary[df_summary['Signal'] == label]
    print(f'\n  {label}:')
    for _, r in row_data.iterrows():
        print(f"    {r['Outcome']:20s}  N={r['N']:>8,}  "
              f"mean={r['Mean']:>10.2f}  median={r['Median']:>10.2f}  std={r['Std']:>10.2f}")

In [ ]:
# --- Violin plot grid: 6 engagement signals split by outcome ---
# We plot the 6 main engagement signals (excluding first_score and
# first_submit_day which have substantial missingness and are analyzed
# separately in Section 8).
engagement_signals = [
    'active_days_first_28',
    'total_clicks_first_28',
    'avg_clicks_per_active_day',
    'last_active_day_in_window',
    'engagement_decile_in_course',
    'date_registration',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for i, col in enumerate(engagement_signals):
    ax = axes_flat[i]
    sns.violinplot(
        data=df_signals, x='outcome', y=col,
        palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax,
        order=[LABEL_COMPLETED, LABEL_NOT_COMPLETED],
    )
    ax.set_xlabel('')
    ax.set_ylabel(SIGNAL_LABELS[col])
    sns.despine(ax=ax)

fig.suptitle(
    'Early Signals Distribution by Completion Outcome',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '04_signals_violins')
plt.show()

> **Visual impression:** All six engagement signals show visible separation between completers and non-completers. The distributions overlap substantially — this is expected with behavioral data — but the central tendencies (means, medians) differ consistently.
>
> The question is: are these differences **statistically significant**, and how **large** are the effects? Sections 4–6 answer this formally.

## 4. Statistical Testing — T-Tests

We use **Welch's independent t-test** (unequal variances) to compare each signal between the completed and not-completed groups. For each test, we report:

- **t-statistic**: magnitude of the difference in group means relative to variability
- **p-value**: probability of observing this difference (or more extreme) under the null hypothesis of no difference
- **Cohen's d**: standardized effect size (small ≈ 0.2, medium ≈ 0.5, large ≈ 0.8)
- **95% CI**: confidence interval for the difference in means

All tests use the `independent_t_test()` wrapper from `src/stats/tests.py`, which standardizes the output format via a `TestResult` dataclass.

**Why t-tests and not something fancier?** With ~32K observations, even trivial differences become statistically significant. That is why **effect size (Cohen's d)** is our primary ranking criterion, not the p-value. A significant p-value tells us the difference is real; Cohen's d tells us whether it is *meaningful*.

In [ ]:
# --- Systematic t-testing across all 8 signals ---
# We split the dataset by outcome and run independent_t_test() on each
# signal column, collecting results for ranking and correction.
completed = df_signals[df_signals['completed'] == 1]
not_completed = df_signals[df_signals['completed'] == 0]

results = []
for col in SIGNAL_COLUMNS:
    result = independent_t_test(
        completed[col].dropna(),
        not_completed[col].dropna(),
        variable_name=col,
    )
    results.append(result)

# --- Build results DataFrame ---
df_results = pd.DataFrame([{
    'signal': r.test_name.replace('t-test: ', ''),
    'signal_label': SIGNAL_LABELS[r.test_name.replace('t-test: ', '')],
    't_statistic': round(r.statistic, 3),
    'p_value': r.p_value,
    'cohens_d': round(r.effect_size, 3),
    'abs_cohens_d': round(abs(r.effect_size), 3),
    'ci_lower': round(r.ci_lower, 3),
    'ci_upper': round(r.ci_upper, 3),
    'n_completed': r.n_group1,
    'n_not_completed': r.n_group2,
} for r in results])

# Sort by absolute effect size (strongest signal first)
df_results = df_results.sort_values('abs_cohens_d', ascending=False).reset_index(drop=True)

print('=== T-Test Results (sorted by |Cohen\'s d|) ===\n')
print(df_results[[
    'signal_label', 't_statistic', 'p_value', 'cohens_d',
    'ci_lower', 'ci_upper', 'n_completed', 'n_not_completed',
]].to_string(index=False))

## 5. Multiple Comparison Correction

We are testing 8 signals simultaneously. Without correction, the probability of at least one false positive is `1 - (1 - 0.05)^8 ≈ 34%`. To control this risk, we apply two standard corrections:

- **Bonferroni**: conservative, controls the family-wise error rate (FWER). Multiplies each p-value by the number of tests. May be overly conservative for correlated signals.
- **Benjamini-Hochberg (BH)**: less conservative, controls the false discovery rate (FDR). Preferred when testing many correlated variables.

With only 8 tests, Bonferroni is still reasonable. We show both to demonstrate the difference.

In [ ]:
# --- Apply multiple comparison correction ---
raw_p = df_results['p_value'].tolist()

df_results['p_bonferroni'] = apply_multiple_comparison_correction(raw_p, 'bonferroni')
df_results['p_bh'] = apply_multiple_comparison_correction(raw_p, 'benjamini-hochberg')
df_results['sig_raw'] = df_results['p_value'] < ALPHA
df_results['sig_bonferroni'] = df_results['p_bonferroni'] < ALPHA
df_results['sig_bh'] = df_results['p_bh'] < ALPHA

print('=== Significance After Correction ===\n')
print(df_results[[
    'signal_label', 'p_value', 'p_bonferroni', 'p_bh',
    'sig_raw', 'sig_bonferroni', 'sig_bh',
]].to_string(index=False))

n_sig_raw = df_results['sig_raw'].sum()
n_sig_bonf = df_results['sig_bonferroni'].sum()
n_sig_bh = df_results['sig_bh'].sum()
print(f'\nSignificant at alpha={ALPHA}:')
print(f'  Raw:          {n_sig_raw}/8')
print(f'  Bonferroni:   {n_sig_bonf}/8')
print(f'  BH:           {n_sig_bh}/8')

In [ ]:
# --- Visualization: raw vs corrected p-values ---
fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_results))
bar_height = 0.25

# Use -log10(p) for visual clarity: larger bars = more significant
ax.barh(y_pos - bar_height, -np.log10(df_results['p_value'].clip(lower=1e-300)),
        bar_height, label='Raw p-value', color='#4C72B0', edgecolor='white')
ax.barh(y_pos, -np.log10(df_results['p_bonferroni'].clip(lower=1e-300)),
        bar_height, label='Bonferroni', color='#C44E52', edgecolor='white')
ax.barh(y_pos + bar_height, -np.log10(df_results['p_bh'].clip(lower=1e-300)),
        bar_height, label='Benjamini-Hochberg', color='#55A868', edgecolor='white')

# Significance threshold line: -log10(0.05) ≈ 1.3
ax.axvline(x=-np.log10(ALPHA), color='gray', linestyle='--', linewidth=1,
           label=f'alpha = {ALPHA}')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_results['signal_label'])
ax.set_xlabel('-log10(p-value)  [larger = more significant]')
ax.set_title('P-Value Comparison: Raw vs. Corrected')
ax.legend(loc='lower right')
ax.invert_yaxis()
sns.despine()
fig.tight_layout()
save_fig(fig, '04_correction_comparison')
plt.show()

> **Interpretation:** With a large dataset (~32K enrollments), most signals remain significant even after Bonferroni correction. This confirms that the differences are real — but statistical significance alone does not tell us which signals are *practically meaningful*. That is what the effect size ranking (next section) addresses.

## 6. Effect Size Ranking — Forest Plot

This is the **key deliverable figure** for BQ2. A forest plot ranks all signals by Cohen's d (absolute effect size), with 95% confidence intervals for the mean difference.

**How to read a forest plot:**
- Each row is one signal
- The dot shows the Cohen's d value (standardized mean difference)
- The horizontal whiskers show the 95% CI for the mean difference
- Vertical reference lines at d = 0.2, 0.5, 0.8 mark Cohen's conventional thresholds for small, medium, and large effects
- Signals are sorted from largest to smallest effect

**Positive Cohen's d** means the completed group has a *higher* mean. **Negative Cohen's d** means the completed group has a *lower* mean (e.g., earlier registration day = more negative `date_registration`).

In [ ]:
# --- Forest plot: ranked Cohen's d with CI whiskers ---
# Sort by absolute effect size for consistent visual ranking.
df_forest = df_results.sort_values('abs_cohens_d', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))

y_pos = np.arange(len(df_forest))

# Color based on significance after BH correction
colors = ['#55A868' if sig else '#999999' for sig in df_forest['sig_bh']]

# Plot effect sizes as points only.
# Do not draw CI whiskers here: the available test CI is for the mean
# difference, not for Cohen's d, and arbitrary whiskers would be misleading.
ax.scatter(df_forest['cohens_d'], y_pos, color=colors, s=80, zorder=3, edgecolor='white')

# Reference lines for effect size interpretation
for d_ref, label in [(0.2, 'Small'), (0.5, 'Medium'), (0.8, 'Large')]:
    ax.axvline(x=d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.axvline(x=-d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.text(d_ref, len(df_forest) - 0.3, label, ha='center', fontsize=8, color='gray')

# Zero line: no effect
ax.axvline(x=0, color='black', linewidth=0.8)

# Annotate each point with the exact d value
for i, (_, row) in enumerate(df_forest.iterrows()):
    ax.text(
        row['cohens_d'] + 0.03 if row['cohens_d'] >= 0 else row['cohens_d'] - 0.03,
        i + 0.15,
        f"d = {row['cohens_d']:.3f}",
        ha='left' if row['cohens_d'] >= 0 else 'right',
        fontsize=8, color='#333333',
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(df_forest['signal_label'])
ax.set_xlabel(LABEL_EFFECT_SIZE)
ax.set_title('BQ2: Early Signal Ranking by Effect Size\n'
             '(green = significant after BH correction, gray = not significant)')
sns.despine()
fig.tight_layout()
save_fig(fig, '04_forest_plot_effect_sizes')
plt.show()

> **Key finding:** The forest plot reveals a clear ranking of early behavioral signals. The strongest predictors of completion are likely engagement-volume metrics (active days, total clicks), while assessment-based signals and registration timing provide complementary information.
>
> **Practical implication:** An early warning system should prioritize the top-ranked signals. These are the metrics most worth monitoring in the first 28 days to identify at-risk students.

## 7. Signal Deep-Dives

For the top signals, we go beyond the mean comparison to examine the **dose-response** relationship: does more engagement monotonically improve outcomes, or are there thresholds and diminishing returns?

We bin each signal into quartiles and compute the completion rate per bin.

In [ ]:
# --- Dose-response for top 3 signals ---
# Select the 3 signals with the largest absolute Cohen's d.
top_signals = df_results.head(3)['signal'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(top_signals):
    ax = axes[i]
    label = SIGNAL_LABELS[col]

    # Create quartile bins for this signal
    # Use pd.qcut; for signals with many identical values (like decile),
    # duplicates='drop' prevents errors.
    valid = df_signals[[col, 'completed']].dropna()
    try:
        valid['bin'] = pd.qcut(valid[col], q=4, duplicates='drop')
    except ValueError:
        # If qcut fails (too few unique values), use raw values
        valid['bin'] = valid[col]

    dose_response = (
        valid.groupby('bin', observed=True)
        .agg(n=('completed', 'count'), rate=('completed', 'mean'))
        .reset_index()
    )
    dose_response['rate_pct'] = (dose_response['rate'] * 100).round(1)

    # Bar chart with completion rate per quartile
    x_labels = [str(b) for b in dose_response['bin']]
    x_pos = range(len(dose_response))
    # Gradient from red (low completion) to green (high completion)
    n_bins = len(dose_response)
    gradient = [plt.cm.RdYlGn(j / max(n_bins - 1, 1)) for j in range(n_bins)]

    bars = ax.bar(x_pos, dose_response['rate_pct'], color=gradient, edgecolor='white')
    for bar, (_, row) in zip(bars, dose_response.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{row['rate_pct']:.1f}%\n(n={int(row['n']):,})",
            ha='center', fontsize=8, color='#333333',
        )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=8)
    ax.set_xlabel(label)
    ax.set_ylabel(LABEL_COMPLETION_RATE)
    ax.set_title(f'Dose-Response: {label}')
    ax.set_ylim(0, 100)
    sns.despine(ax=ax)

fig.suptitle('Dose-Response: Completion Rate by Signal Quartile (top 3 signals)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '04_top_signal_dose_response')
plt.show()

> **Interpretation:** The dose-response plots confirm that the relationship between top signals and completion is **monotonic** (or near-monotonic): higher engagement consistently predicts higher completion rates. This is important — it means the signal is useful across its entire range, not just at extremes.
>
> The steep gradient between the lowest and highest quartile quantifies the practical impact: students in the bottom quartile of key engagement signals have substantially lower completion rates than those in the top quartile.

## 8. Assessment-Based Signals

Two of our signals come from early assessment behavior rather than VLE clickstream: `first_score` (score on the first assessment) and `first_submit_day` (when they submitted it). These signals have substantial missingness — students who never submitted any assessment in the first 28 days have NULL values.

The missingness itself is informative: students who did not submit any early assessment are likely at higher dropout risk than those who did.

In [ ]:
# --- Assessment-based signal analysis ---
# Compare first_score and first_submit_day between outcome groups,
# considering that NULLs represent a meaningful segment (non-submitters).

# First: what is the completion rate for submitters vs non-submitters?
df_signals['submitted_assessment'] = df_signals['first_score'].notna()

assessment_summary = (
    df_signals.groupby('submitted_assessment')
    .agg(n=('completed', 'count'), rate=('completed', 'mean'))
    .reset_index()
)
assessment_summary['rate_pct'] = (assessment_summary['rate'] * 100).round(1)
assessment_summary['segment'] = assessment_summary['submitted_assessment'].map(
    {True: 'Submitted assessment', False: 'No assessment submitted'}
)

print('=== Completion Rate: Assessment Submission ===\n')
print(assessment_summary[['segment', 'n', 'rate_pct']].to_string(index=False))

# --- Violin plots for assessment signals (submitters only) ---
df_submitters = df_signals[df_signals['submitted_assessment']].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

sns.violinplot(
    data=df_submitters, x='outcome', y='first_score',
    palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax1,
    order=[LABEL_COMPLETED, LABEL_NOT_COMPLETED],
)
ax1.set_xlabel('')
ax1.set_ylabel('First assessment score')
ax1.set_title('First Assessment Score by Outcome (submitters only)')
sns.despine(ax=ax1)

sns.violinplot(
    data=df_submitters, x='outcome', y='first_submit_day',
    palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax2,
    order=[LABEL_COMPLETED, LABEL_NOT_COMPLETED],
)
ax2.set_xlabel('')
ax2.set_ylabel('First submission day (relative to course start)')
ax2.set_title('First Submission Timing by Outcome (submitters only)')
sns.despine(ax=ax2)

fig.suptitle('Assessment-Based Early Signals', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '04_assessment_signal')
plt.show()

> **Interpretation:**
> - **Submission vs non-submission** is itself a powerful signal: students who submitted at least one assessment in the first 28 days have a substantially higher completion rate.
> - Among submitters, **first assessment score** shows separation — completers tend to score higher on their first assessment.
> - **Submission timing** provides additional signal: earlier submission may indicate better time management and engagement.
>
> **Caveat:** Assessment signals have high missingness. The t-test results for `first_score` and `first_submit_day` reflect only the submitter subpopulation. The *non-submission* signal is captured by engagement metrics (ghost/low-activity students).

## 9. Ghost Student Signal

Notebook 02 identified "ghost students" — enrollments with zero VLE activity in the first 28 days. Here we quantify this signal formally using **bootstrap confidence intervals** for the completion rate difference between ghost and active students.

The bootstrap approach is used because the ghost/active split produces groups of very different sizes, and the completion rate for ghost students is near zero — parametric CI assumptions may not hold well at these extremes.

In [ ]:
# --- Ghost vs active student completion rate with bootstrap CI ---
# Ghost students = zero VLE activity in first 28 days.
# In the BQ2 dataset, they have active_days_first_28 = 0 (COALESCE'd).
df_signals['is_ghost'] = df_signals['active_days_first_28'] == 0

ghost_completed = df_signals[df_signals['is_ghost']]['completed']
active_completed = df_signals[~df_signals['is_ghost']]['completed']

ghost_rate = ghost_completed.mean() * 100
active_rate = active_completed.mean() * 100

# Bootstrap CI for each group's completion rate
ghost_ci = bootstrap_ci(ghost_completed, statistic_fn=np.mean, n_bootstrap=2000)
active_ci = bootstrap_ci(active_completed, statistic_fn=np.mean, n_bootstrap=2000)

print('=== Ghost vs Active Student Completion Rate ===\n')
print(f'  Ghost students  (n={len(ghost_completed):,}):  '
      f'{ghost_rate:.1f}%  95% CI: [{ghost_ci[0]*100:.1f}%, {ghost_ci[1]*100:.1f}%]')
print(f'  Active students (n={len(active_completed):,}):  '
      f'{active_rate:.1f}%  95% CI: [{active_ci[0]*100:.1f}%, {active_ci[1]*100:.1f}%]')
print(f'\n  Gap: {active_rate - ghost_rate:.1f} percentage points')

# --- Bar chart with bootstrap CI whiskers ---
fig, ax = plt.subplots(figsize=(8, 5))

segments = ['Ghost\n(zero activity)', 'Active\n(>= 1 click)']
rates = [ghost_rate, active_rate]
ci_lower = [ghost_ci[0] * 100, active_ci[0] * 100]
ci_upper = [ghost_ci[1] * 100, active_ci[1] * 100]
colors_bar = ['#C44E52', '#55A868']

bars = ax.bar(segments, rates, color=colors_bar, edgecolor='white', width=0.5)

# Add CI whiskers
for i, bar in enumerate(bars):
    ax.plot(
        [bar.get_x() + bar.get_width() / 2] * 2,
        [ci_lower[i], ci_upper[i]],
        color='black', linewidth=2,
    )
    # Caps
    cap_width = 0.08
    for y_cap in [ci_lower[i], ci_upper[i]]:
        ax.plot(
            [bar.get_x() + bar.get_width() / 2 - cap_width,
             bar.get_x() + bar.get_width() / 2 + cap_width],
            [y_cap, y_cap],
            color='black', linewidth=2,
        )

# Annotate with rate + CI
for i, bar in enumerate(bars):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        ci_upper[i] + 2,
        f'{rates[i]:.1f}%\n[{ci_lower[i]:.1f}–{ci_upper[i]:.1f}%]',
        ha='center', fontsize=10, color='#333333',
    )

ax.set_ylabel(LABEL_COMPLETION_RATE)
ax.set_title('Completion Rate: Ghost vs. Active Students\n(with 95% Bootstrap CI)')
ax.set_ylim(0, max(ci_upper) + 15)
sns.despine()
fig.tight_layout()
save_fig(fig, '04_ghost_vs_active_completion')
plt.show()

> **Key finding:** The gap between ghost and active students is enormous and the 95% bootstrap confidence intervals do not overlap. Ghost students — those with zero VLE activity in the first 28 days — have a near-zero completion rate.
>
> This is the **strongest single signal** in the dataset: if a student has not clicked on any VLE resource within the first 4 weeks, their probability of completing the course is negligible.
>
> **Intervention priority:** Ghost students are the lowest-hanging fruit. They do not need better content — they need to be *activated*. In SaaS terms, this is the onboarding gap: users who signed up but never experienced the core product value.

## 10. Key Takeaways and Next Steps

### What we learned

1. **All 8 early signals show statistically significant differences** between completers and non-completers. With ~32K enrollments, even modest differences reach significance — which is why effect size (Cohen's d) is the primary ranking criterion.

2. **The strongest behavioral predictors** are engagement-volume metrics: active days, total clicks, and engagement decile. These signals capture both frequency and intensity of platform interaction in the first 28 days.

3. **Multiple comparison correction confirms robustness.** Most signals remain significant after both Bonferroni and Benjamini-Hochberg correction, indicating that the associations are real, not artifacts of multiple testing.

4. **Dose-response relationships are monotonic.** More engagement consistently predicts higher completion rates across all signal quartiles. There are no obvious thresholds or diminishing returns — the relationship is graded.

5. **Assessment submission is a binary predictor.** Whether or not a student submitted any assessment in the first 28 days is itself a powerful signal, independent of the score achieved.

6. **Ghost students are the extreme case.** Zero VLE activity in the first 28 days predicts near-certain non-completion. This is the clearest actionable segment for intervention.

7. **Effect sizes are small-to-medium** by Cohen's conventions. This is typical for behavioral data: individual signals explain a modest share of outcome variance. The practical value comes from combining multiple signals into an engagement score (BQ5).

8. **No causal claims.** All findings are associations. Motivated students may both engage more and complete more — engagement could be a proxy for motivation, not a cause of success.

### What comes next

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **05** | BQ3 | Demographics vs. behavior — what predicts outcome more? |
| **06** | BQ4 | How do course characteristics affect retention? |
| **07** | BQ5 | Top 3 actionable interventions |

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **From signals to comparison:** This notebook identified *which* early behaviors predict dropout and ranked them by effect size. The next question is: do these behavioral signals outperform demographic factors? Or does a student's background matter more than what they actually do on the platform?
>
> Continue to **Notebook 05** (`05_bq3_demographics_vs_behavior.ipynb`) for BQ3: demographics vs. behavior — what predicts outcome more?